In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

N = 1000

first_names = ["John","Alice","Ravi","Meera","David","Sara","Arjun","Priya","Karthik","Nina","Omar","Li","Chen","Asha","Miguel"]
last_names  = ["Smith","Patel","Kumar","Lee","Johnson","Brown","Singh","Iyer","Garcia","Nguyen","Shah","Rao","Das","Wilson","Kim"]

def rand_name():
    if random.random() < 0.05:  # 5% missing
        return None
    return f"{random.choice(first_names)} {random.choice(last_names)}"

def rand_email(name):
    if random.random() < 0.12:  # 12% missing
        return None
    if random.random() < 0.05:  # 5% invalid
        return "bad-email@@"
    user = "".join(name.lower().split()) if name else "user"
    domain = random.choice(["gmail.com", "yahoo.com", "company.com", "outlook.com"])
    return f"{user}{random.randint(1,999)}@{domain}"

def rand_country():
    # intentionally inconsistent variants
    return random.choice(["USA","US","United States","india","India","IN","U.S.","United States of America", None])

def rand_signup_date():
    # mix valid and invalid formats
    base = datetime(2020,1,1) + timedelta(days=random.randint(0, 1500))
    fmt = random.choice(["%Y-%m-%d","%d-%m-%Y","%d/%m/%Y","%Y/%m/%d","%m-%d-%y"])
    val = base.strftime(fmt)

    if random.random() < 0.06:  # 6% invalid dates
        return "2021-13-01"
    if random.random() < 0.03:  # 3% empty
        return None
    return val

def rand_credit_score():
    if random.random() < 0.10:  # 10% missing
        return None
    if random.random() < 0.03:  # 3% non-numeric
        return "abc"
    # sometimes out of range
    if random.random() < 0.03:
        return random.choice([200, 999])
    return int(np.random.normal(680, 60))

def rand_last_updated():
    base = datetime(2022,1,1) + timedelta(days=random.randint(0, 1000))
    if random.random() < 0.05:
        return "invalid_date"
    # mix formats
    return base.strftime(random.choice(["%Y-%m-%d","%Y/%m/%d","%d-%m-%Y"]))

def rand_status():
    return random.choice(["ACTIVE","INACTIVE","SUSPENDED", "active", None])

def rand_kyc():
    return random.choice(["VERIFIED","PENDING","FAILED", "verified", None])

def rand_currency():
    return random.choice(["USD","INR","usd","INR ", "EURO", None])  # EURO invalid on purpose

def rand_balance():
    if random.random() < 0.05:
        return None
    # introduce negatives sometimes
    bal = round(random.uniform(-200, 5000), 2)
    return bal

# Create base unique-ish customer_ids
customer_ids = list(range(10000, 10000 + N))
rows = []
for cid in customer_ids:
    name = rand_name()
    rows.append({
        "customer_id": cid,
        "full_name": name,
        "email": rand_email(name),
        "country": rand_country(),
        "signup_date": rand_signup_date(),
        "credit_score": rand_credit_score(),
        "account_status": rand_status(),
        "kyc_status": rand_kyc(),
        "last_updated": rand_last_updated(),
        "balance": rand_balance(),
        "currency": rand_currency()
    })

df = pd.DataFrame(rows)

# Add duplicates (5% duplicate customer_id, with potentially different fields)
dup_count = int(0.05 * N)
dup_rows = df.sample(dup_count, random_state=42).copy()
# mutate duplicate rows to simulate updates
dup_rows["last_updated"] = dup_rows["last_updated"].apply(lambda _: rand_last_updated())
dup_rows["balance"] = dup_rows["balance"].apply(lambda _: rand_balance())
dup_rows["kyc_status"] = dup_rows["kyc_status"].apply(lambda _: rand_kyc())
df = pd.concat([df, dup_rows], ignore_index=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head(), len(df)


(   customer_id      full_name                      email        country  \
 0        10352     Asha Kumar   ashakumar185@company.com  United States   
 1        10689       John Rao     johnrao255@company.com           U.S.   
 2        10485  Miguel Wilson  miguelwilson940@gmail.com           U.S.   
 3        10388            NaN          user875@gmail.com             IN   
 4        10031    Meera Brown  meerabrown570@company.com            USA   
 
   signup_date credit_score account_status kyc_status  last_updated  balance  \
 0  07/11/2022          200         active   VERIFIED    10-09-2022  1509.02   
 1  2022-05-16          abc         active   VERIFIED    2023-01-13  1786.50   
 2  06-06-2022          612         active   VERIFIED    13-10-2022  2243.83   
 3  2022/06/04          679       INACTIVE   VERIFIED    12-10-2022  -173.17   
 4  24-11-2022          594         active   verified  invalid_date  1902.89   
 
   currency  
 0      usd  
 1      INR  
 2      NaN  
 3  

In [2]:
df.to_csv("../data/legacy_customers.csv", index=False)
print("Saved:", "../data/legacy_customers.csv", "rows:", len(df))


Saved: ../data/legacy_customers.csv rows: 1050
